# UPDATED FEB 2026: Classify bgcArgo and SOCAT (P1-processed) using core clustering results (P2-clustered)

- Remove SIZ here
Save outputs in P2-clustered/

In [178]:
from importlib import reload
import mod_loading as loader
import mod_pcm
import datetime 
from datetime import datetime
import xarray as xr
import pandas as pd

In [179]:
import numpy as np

import figs_pcm


import mod_preprocessing as mod_prep
import mod_regression as mod_reg
import mod_pcm 

In [180]:
### SET NOTEBOOK RUN PARAMETERS ### ===================

n_pca = 8               # PC components 
n_gmm = 6               # GMM classes 
dbar_limit = 501        # Depth limit for PCA 

# ================================================

# use_preprocessed = False # Set to True if you want to load pre-centered data 
save_run = True

pcm_params = 'pc' + str(n_pca) + '_gmm' + str(n_gmm)
save_path = '../working-vars/regression/P2-clustered/' + pcm_params + '/'
datetag = datetime.now().strftime('%Y%m%d') # For filenames

# Add GMM probabilities and clustering assignments to the ML inputs



In [ ]:
# moved to mod_pcm
# def add_class_assignments(platDF, PCM_finder, n_gmm, profid_col='nearest_profid'):
#     """ 
    
#     @param  profid_col: 'profid' for floatDF, 'nearest_profid' for shipDF 
#             PCM_finder: dataframe with profid as index and columns for each class probability and 'class_assign' for assigned class

#             [Y_gmm, allprobs] = loader.import_p2_clustered(pcm_params = pcm_params)
#             PCM_finder = allprobs.set_index('profid') # temp object to search
#             PCM_finder['class_assign'] = Y_gmm.loc[PCM_finder.index, 'class']

#     """
#     platDF = platDF.copy()
#     for k in range(1, n_gmm+1):
#         platDF['class' + str(k) + '_prob'] = platDF[profid_col].apply(lambda x: PCM_finder.loc[x, str(k)] if x in PCM_finder.index else np.nan)
#     platDF['class_assign'] = (platDF[profid_col].apply(lambda x: PCM_finder.loc[x, 'class_assign'] if x in PCM_finder.index else np.nan))
#     # platDF['prof_datetag'] = platDF['prof_datetag'].astype(str)

#     # platDF = platDF.dropna(subset=['class_assign'])
#     # print('Initial classes assigned. Filling missing values where adjacent classes match...')

# #     # Fill in missing  cluster values where we can -- 
# #     # look for same float, profiles within +/- 11 days
# #     # If cluster was same before and after, assign to that cluster

    
#     return platDF

In [ ]:
# # moved to mod_pcm
# def fill_missing_float_classes(argoDF):
#     """ for soccom 20 m aves
#     doesn't work for shipDF yet"""
#     # argoDF['prof_datetag'] = argoDF['prof_datetag'].astype(str)
#     # argoDF['datetime'] = pd.to_datetime(argoDF['datetime'])

#     # argoDF = argoDF.set_index('profid')
#     argoDF_nona = argoDF.dropna(subset='class_assign')
#     missing = argoDF[argoDF.class_assign.isna()].reset_index()
#     available_prof_datetags = argoDF_nona.prof_datetag.unique().astype(str)
#     print('Initially missing class assignments for ' + str(len(missing)) + ' profiles, searching to fill with adjacent datetags...')

#     profsReplaced = []

#     for ind, tag in enumerate(missing.profid.unique()):
#         obs = missing.iloc[ind, :]
#         same_float = argoDF_nona[argoDF_nona.wmoid==obs.wmoid].copy()

#         if len(same_float)>0:
#             same_float.loc[:,'timediff'] = same_float.apply(lambda row: abs((row.datetime - (obs.datetime)).days), axis=1)
#             valid = same_float[same_float.timediff<12]
#             if valid.class_assign.nunique() == 1:
#                 argoDF.loc[tag, 'class_assign'] = valid.class_assign.values[0]
#                 profsReplaced = profsReplaced + [tag]

#     argoDF = argoDF.dropna(subset='class_assign')
#     print('Filled in ' + str(len(profsReplaced)) + ' missing assignments using adjacent profiles; total ' + str(len(argoDF)) + ' valid profiles')

#     return argoDF 

In [ ]:
# bgcDS_test['profid'] = bgcDS_test['profid'].apply(lambda x: x.replace('id', 'testid'))

# temp = bgcDS_test.to_dataframe().reset_index().copy()
# temp['profid'] = temp['profid'].apply(lambda x: x.replace('id', 'testid'))
# bgcDS_test_updated = temp.set_index(['profid', 'pressure']).to_xarray()
# bgcDS_test_updated.assign_coords()

ValueError: the first argument to .assign_coords must be a dictionary

In [ ]:
# [coreDS, coreINDEX] = loader.import_core_data(type='processed') # has adt, sla, mld, pco2

# Import PCM results from 2.1

In [181]:
# ====== New version from feb 11 with import p2 data
[bgcArgo_trainval, bgcArgo_test, socat_trainval, coreArgo_application] = loader.import_p1_processed()
[PCM_components, PCM_finder] = loader.import_p2_clustered(type='pcm_probs', pcm_params = pcm_params)


Importing processed dataframes...
Returned [bgcArgo_trainval(2014-2023), bgcArgo_test(2024), socat_trainval(2014-2023), coreArgo_application(2014-2023)]
Importing clustering results for pc8_gmm6...
Returned [PCM_components, PCM_finder (probabilities)]
Returned [bgcArgo_trainval(2014-2023), bgcArgo_test(2024), socat_trainval(2014-2023), coreArgo_application(2014-2023)]


In [182]:
PCM_finder

,1,2,3,4,5,6,class_assign,prof_datetag
profid,,,,,,,,
3901116_id230,1.221626e-265,9.999952e-01,5.005922e-09,4.761933e-06,8.421335e-107,7.780087e-23,2,3901116_20200423
7900672_id070,1.504950e-02,1.723851e-08,9.965891e-16,1.216886e-09,9.849505e-01,4.748475e-38,5,7900672_20180301
6901650_id030,9.951652e-01,8.533714e-10,2.375880e-09,2.827776e-03,1.464748e-09,2.006978e-03,1,6901650_20150605
5900704_id490,2.317572e-297,1.000000e+00,1.258162e-11,6.068232e-09,7.383719e-116,3.829733e-36,2,5900704_20190601
7900632_id084,6.506999e-06,3.443722e-10,1.436543e-20,1.093385e-10,9.999935e-01,2.111749e-46,5,7900632_20210427
...,...,...,...,...,...,...,...,...
5904897_id172,5.882859e-155,1.638245e-01,8.360185e-01,1.569834e-04,9.376667e-63,2.179208e-11,3,5904897_20190623
5905772_id081,9.999826e-01,7.807351e-11,1.413136e-10,1.696064e-05,1.753216e-09,4.660801e-07,1,5905772_20201101
5905215_id009,9.993955e-01,1.778084e-08,3.154096e-12,1.742427e-05,5.870136e-04,6.370674e-17,1,5905215_20180309


# Classify floats (bgcArgo_trainval, bgcArgo_test, coreArgo_application)

In [115]:
# Add class assignments (will not drop nans yet)
floatDF = mod_pcm.add_class_assignments(bgcArgo_trainval, PCM_finder, n_gmm, profid_col='profid')
floatDF['prof_datetag'] = floatDF['prof_datetag'].astype(str)
floatDF_filled = mod_pcm.fill_missing_float_classes(floatDF.set_index('profid'), n_gmm=n_gmm)



# coreArgo_application['prof_datetag'] = coreArgo_application['profid'].apply(lambda x: str(x).split('_')[0])
# coreArgo_application['prof_datetag'] = coreArgo_application.apply(lambda row: row.prof_datetag + '_' + row.datetime.strftime('%Y%m%d'), axis=1)

Initially missing class assignments for 267 profiles, searching to fill with adjacent datetags...
Filled in 227 missing assignments using adjacent profiles; total 10781 valid profiles


In [112]:
testDF = mod_pcm.add_class_assignments(bgcArgo_test, PCM_finder, n_gmm, profid_col='profid')
testDF['prof_datetag'] = testDF['prof_datetag'].astype(str)
testDF_filled = mod_pcm.fill_missing_float_classes(testDF.set_index('profid'), n_gmm=n_gmm)

Initially missing class assignments for 6 profiles, searching to fill with adjacent datetags...
Filled in 4 missing assignments using adjacent profiles; total 2111 valid profiles


In [119]:
coreDF = mod_pcm.add_class_assignments(coreArgo_application, PCM_finder, n_gmm, profid_col='profid')
coreDF = (mod_prep.index_by_prof_datetag(coreDF)
                .reset_index().set_index('profid') # add prof_datetag as column so you can fill in 
)
coreDF_filled = mod_pcm.fill_missing_float_classes(coreDF, n_gmm=n_gmm)


Initially missing class assignments for 8237 profiles, searching to fill with adjacent datetags...
Filled in 5113 missing assignments using adjacent profiles; total 322997 valid profiles


In [174]:
floatDF_filled

,wmoid,prof_datetag,datetime,latitude,longitude,ydcos,ydsin,CT,SA,sst,...,pco2_atmos,pco2_ocean,delta_pco2,class1_prob,class2_prob,class3_prob,class4_prob,class5_prob,class6_prob,class_assign
profid,,,,,,,,,,,,,,,,,,,,,
5904187_id002,5904187,5904187_20140424,2014-04-24 01:35:59.999999,-45.027,209.980,-0.365551,0.930791,13.499859,34.489418,13.495183,...,382.385383,359.594018,-22.791365,8.203667e-53,3.536559e-04,2.983103e-06,9.766923e-01,2.724776e-30,2.295110e-02,4.0
5904187_id003,5904187,5904187_20140429,2014-04-29 10:21:59.999998,-44.925,209.465,-0.449781,0.893139,13.255945,34.504120,13.251828,...,392.294217,356.315818,-35.978399,7.441197e-53,3.564523e-04,1.752034e-05,9.277201e-01,2.379849e-30,7.190595e-02,4.0
5904187_id004,5904187,5904187_20140504,2014-05-04 17:40:00.000001,-45.037,209.571,-0.529291,0.848440,13.192600,34.513058,13.188741,...,390.099938,358.544402,-31.555536,1.056995e-53,6.353372e-04,2.556170e-05,9.940579e-01,3.934667e-30,5.281248e-03,4.0
5904187_id005,5904187,5904187_20140510,2014-05-10 00:45:59.999996,-44.769,209.377,-0.604283,0.796770,13.069445,34.476292,13.064869,...,386.600808,360.784113,-25.816695,1.600710e-52,4.279380e-04,3.638408e-05,9.424771e-01,6.256914e-30,5.705855e-02,4.0
5904187_id006,5904187,5904187_20140515,2014-05-15 07:09:00.000005,-44.989,209.152,-0.673884,0.738837,12.586998,34.523502,12.583867,...,393.854081,355.840425,-38.013657,9.509072e-52,3.834575e-04,1.380343e-03,5.437665e-01,2.305698e-29,4.544697e-01,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5906583_id030,5906583,5906583_20231110,2023-11-10 05:56:00.000004,-62.603,163.310,0.622491,-0.782627,-1.736849,34.314301,-1.740603,...,411.476586,422.057030,10.580444,2.572998e-03,5.414120e-10,1.832177e-19,9.920684e-13,9.974270e-01,3.154103e-51,5.0
5906583_id031,5906583,5906583_20231120,2023-11-20 12:59:00.000000,-62.689,162.887,0.750620,-0.660735,-1.672186,34.215834,-1.676101,...,416.243436,408.826105,-7.417331,1.176118e-03,3.204954e-10,8.265312e-19,1.275828e-12,9.988239e-01,3.729968e-49,5.0
5906583_id032,5906583,5906583_20231130,2023-11-30 19:55:59.999998,-62.720,162.534,0.855236,-0.518240,-1.683684,34.142183,-1.687714,...,410.705222,401.951619,-8.753604,2.650822e-05,1.856321e-10,2.257993e-19,9.136115e-13,9.999735e-01,4.952915e-51,5.0


In [173]:
testDF_filled

,wmoid,prof_datetag,datetime,latitude,longitude,ydcos,ydsin,CT,SA,sst,...,pco2_atmos,pco2_ocean,delta_pco2,class1_prob,class2_prob,class3_prob,class4_prob,class5_prob,class6_prob,class_assign
profid,,,,,,,,,,,,,,,,,,,,,
1902494_testid005,1902494,1902494_20240414,2024-04-14 01:05:59.999995,-46.051,8.418,-0.208893,0.977938,8.362248,33.980043,8.352377,...,412.286438,372.484853,-39.801585,1.702776e-04,1.094715e-08,3.423930e-11,0.911831,2.618179e-16,0.087999,4.0
1902494_testid006,1902494,1902494_20240415,2024-04-15 00:34:59.999996,-46.079,8.489,-0.225324,0.974284,8.462328,33.977941,8.452329,...,411.843348,377.974947,-33.868401,1.781132e-04,1.104661e-08,4.066591e-11,0.777059,4.300568e-16,0.222763,4.0
1902494_testid006,1902494,1902494_20240415,2024-04-15 23:15:59.999996,-46.119,8.556,-0.241134,0.970492,8.386343,33.973935,8.376351,...,412.895518,377.332637,-35.562880,1.781132e-04,1.104661e-08,4.066591e-11,0.777059,4.300568e-16,0.222763,4.0
1902494_testid008,1902494,1902494_20240416,2024-04-16 21:53:00.000002,-46.144,8.612,-0.256834,0.966456,8.339002,33.972940,8.329040,...,413.568473,374.948496,-38.619977,1.584007e-04,1.041332e-08,1.969851e-10,0.504112,8.022330e-16,0.495730,4.0
1902494_testid009,1902494,1902494_20240417,2024-04-17 20:03:00.000003,-46.172,8.652,-0.272156,0.962253,8.574900,33.975963,8.564762,...,413.207793,377.168868,-36.038925,5.642835e-04,7.678293e-09,1.983415e-11,0.825896,8.464087e-17,0.173540,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7902134_testid001,7902134,7902134_20241202,2024-12-02 05:03:00.000003,-43.610,266.023,0.873645,-0.486564,11.344387,34.209100,11.335782,...,411.264266,407.282639,-3.981627,2.372731e-18,2.203116e-07,2.805410e-08,0.517884,3.405140e-23,0.482116,4.0
7902134_testid002,7902134,7902134_20241212,2024-12-12 10:44:59.999997,-43.667,266.415,0.945378,-0.325975,12.071960,34.221409,12.062844,...,413.797331,445.396625,31.599294,4.451204e-19,2.398249e-07,5.592358e-09,0.736659,1.066122e-23,0.263341,4.0
7902134_testid003,7902134,7902134_20241222,2024-12-22 16:30:59.999996,-43.583,266.845,0.987875,-0.155254,12.017707,34.200234,12.008207,...,414.398989,458.409796,44.010807,2.517224e-18,1.817059e-07,1.416120e-08,0.412730,6.674805e-23,0.587270,6.0


In [120]:
coreDF_filled

,prof_datetag,level_0,index,pressure,wmoid,latitude,longitude,datetime,yearday,CT,...,sss,vapor_pres_atm,pco2_atmos,class1_prob,class2_prob,class3_prob,class4_prob,class5_prob,class6_prob,class_assign
profid,,,,,,,,,,,,,,,,,,,,,
1900410_id260,1900410_20140102,0,0,6.4,1900410.0,-40.358000,95.357000,2014-01-02 02:34:12,1.107083,12.707135,...,35.199985,0.014217,392.043076,9.191684e-120,1.533619e-03,9.984318e-01,0.000035,2.225593e-51,6.804707e-09,3.0
1900410_id261,1900410_20140112,1,1,7.0,1900410.0,-40.109000,96.083000,2014-01-12 06:35:16,11.274491,13.136123,...,35.198965,0.014621,391.075263,2.591259e-124,2.979988e-03,9.969694e-01,0.000051,1.209735e-52,5.030543e-09,3.0
1900410_id262,1900410_20140122,2,2,6.0,1900410.0,-40.247000,96.676000,2014-01-22 10:36:20,21.441898,14.351957,...,35.140420,0.015823,386.076861,2.446410e-133,4.037185e-02,9.594548e-01,0.000173,1.000042e-54,1.178668e-11,3.0
1900410_id263,1900410_20140201,3,3,6.6,1900410.0,-40.371000,96.900000,2014-02-01 14:33:32,31.606620,13.675696,...,35.180803,0.015145,388.967172,8.513762e-133,1.212673e-02,9.877921e-01,0.000081,5.429533e-55,2.445882e-11,3.0
1900410_id264,1900410_20140211,4,4,5.9,1900410.0,-40.340000,96.969000,2014-02-11 18:47:10,41.782755,13.617688,...,35.184837,0.015088,393.248530,3.069963e-134,1.350619e-02,9.864227e-01,0.000071,1.828846e-55,4.608275e-11,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7901011_id001,7901011_20231111,328933,328933,10.6,7901011.0,-54.642010,-51.690847,2023-11-11 21:57:00,3601.914583,3.979210,...,34.429393,0.007856,405.579794,9.817612e-01,6.247987e-08,2.267062e-07,0.018206,3.092963e-07,3.209545e-05,1.0
7901011_id002,7901011_20231123,328934,328934,3.1,7901011.0,-53.820388,-49.906383,2023-11-23 17:01:30,3613.709375,4.143979,...,34.404664,0.007947,411.912454,7.824875e-01,1.323220e-07,1.816949e-06,0.074785,1.206493e-08,1.427259e-01,1.0
7901011_id003,7901011_20231203,328935,328935,2.8,7901011.0,-53.981737,-48.127877,2023-12-03 12:41:30,3623.528819,3.454201,...,34.374525,0.007570,405.887082,9.989221e-01,4.142860e-08,9.324309e-07,0.001071,1.940278e-06,3.795500e-06,1.0


In [ ]:
# # bgcArgo_trainval = bgcArgo_trainval.copy(); socat_trainval = socat_trainval.copy()
# for k in range(1, n_gmm+1):
#     bgcArgo_trainval['class' + str(k) + '_prob'] = bgcArgo_trainval['profid'].apply(lambda x: PCM_finder.loc[x, str(k)] if x in PCM_finder.index else np.nan)
#     socat_trainval['class' + str(k) + '_prob'] = socat_trainval['nearest_profid'].apply(lambda x: PCM_finder.loc[x, str(k)] if x in PCM_finder.index else np.nan)

# bgcArgo_trainval['class_assign'] = (bgcArgo_trainval['profid'].apply(lambda x: PCM_finder.loc[x, 'class_assign'] if x in PCM_finder.index else np.nan))
# socat_trainval['class_assign'] = (socat_trainval['nearest_profid'].apply(lambda x: PCM_finder.loc[x, 'class_assign'] if x in PCM_finder.index else np.nan))

# bgcArgo_trainval = bgcArgo_trainval.dropna(subset=['class_assign'])
# socat_trainval = socat_trainval.dropna(subset=['class_assign'])

In [ ]:

# [bgcArgo_trainval, bgcArgo_test, socat_trainval, coreArgo_application] = loader.import_p1_processed()
# [Y_gmm, allprobs] = loader.import_p2_clustered(pcm_params = pcm_params)

# PCM_finder = allprobs.set_index('profid') # temp object to search
# PCM_finder['class_assign'] = Y_gmm.loc[PCM_finder.index, 'class']

# # Add clustering probabilities to the float DF

# # if '0' in PCM_finder.columns: # shouldn't need anymore. fixed in 2.1
# #    PCM_finder = PCM_finder.rename(columns = {str(k): k+1 for k in np.arange(n_gmm-1, -1, -1)})
# # print(PCM_finder.columns)

# # bgcArgo_trainval = bgcArgo_trainval.copy(); socat_trainval = socat_trainval.copy()
# for k in range(1, n_gmm+1):
#     bgcArgo_trainval['class' + str(k) + '_prob'] = bgcArgo_trainval['profid'].apply(lambda x: PCM_finder.loc[x, str(k)] if x in PCM_finder.index else np.nan)
#     socat_trainval['class' + str(k) + '_prob'] = socat_trainval['nearest_profid'].apply(lambda x: PCM_finder.loc[x, str(k)] if x in PCM_finder.index else np.nan)

# bgcArgo_trainval['class_assign'] = (bgcArgo_trainval['profid'].apply(lambda x: PCM_finder.loc[x, 'class_assign'] if x in PCM_finder.index else np.nan))
# socat_trainval['class_assign'] = (socat_trainval['nearest_profid'].apply(lambda x: PCM_finder.loc[x, 'class_assign'] if x in PCM_finder.index else np.nan))

# bgcArgo_trainval = bgcArgo_trainval.dropna(subset=['class_assign'])
# socat_trainval = socat_trainval.dropna(subset=['class_assign'])

# Classify SOCAT

In [184]:
shipDF = mod_pcm.add_class_assignments(socat_trainval, PCM_finder, n_gmm, profid_col='nearest_profid')
shipDF['sample_datetag'] = shipDF['cruiseid'] + '_' + shipDF['datetime'].dt.strftime('%Y%m%d%H')
shipDF_filled = mod_pcm.fill_missing_ship_classes(shipDF.set_index('sample_datetag'), n_gmm)

Filled in 163 missing assignments using adjacent profiles
Total 6139 valid profiles


In [170]:
# missingClasses = shipDF[shipDF['class_assign'].isna()]  #.reset_index().set_index('sample_datetag')
# samplesReplaced = []; 

# for sdt, obs in missingClasses.iterrows(): # for sample_datetag, row ... 
#     same_cruise = shipDF[shipDF['cruiseid'] == obs.cruiseid].copy()

#     if len(same_cruise)>0:
#         same_cruise['timediff'] = same_cruise.apply(lambda row: abs((row.datetime - (obs.datetime)).days), axis=1)
#         valid = same_cruise[same_cruise.timediff<1.5].copy()
    
#         if valid.class_assign.nunique() == 1:

#             shipDF.loc[sdt, 'class_assign'] = valid['class_assign'].values[0]
#             for k in range(1, n_gmm+1):
#                 shipDF.loc[sdt,'class' + str(k) + '_prob'] = valid['class' + str(k) + '_prob'].values[0]
#             samplesReplaced.append(sdt)

# shipDF = shipDF.dropna(subset='class_assign')
# print('Filled in ' + str(len(samplesReplaced)) + ' missing assignments using adjacent profiles')
# print('Total ' + str(len(shipDF)) + ' valid profiles')

Filled in 163 missing assignments using adjacent profiles
Total 6139 valid profiles


In [ ]:
# moved to mod_pcm
# def fill_missing_ship_classes(shipDF, n_gmm):
#     missingClasses = shipDF[shipDF['class_assign'].isna()]  #.reset_index().set_index('sample_datetag')
#     samplesReplaced = []; 

#     for sdt, obs in missingClasses.iterrows(): # for sample_datetag, row ... 
#         same_cruise = shipDF[shipDF['cruiseid'] == obs.cruiseid].copy()

#         if len(same_cruise)>0:
#             same_cruise['timediff'] = same_cruise.apply(lambda row: abs((row.datetime - (obs.datetime)).days), axis=1)
#             valid = same_cruise[same_cruise.timediff<2].copy()
        
#             if valid.class_assign.nunique() == 1:

#                 shipDF.loc[sdt, 'class_assign'] = valid['class_assign'].values[0]
#                 for k in range(1, n_gmm+1):
#                     shipDF.loc[sdt,'class' + str(k) + '_prob'] = valid['class' + str(k) + '_prob'].values[0]
#                 samplesReplaced.append(sdt)

#     shipDF = shipDF.dropna(subset='class_assign')
#     print('Filled in ' + str(len(samplesReplaced)) + ' missing assignments using adjacent profiles')
#     print('Total ' + str(len(shipDF)) + ' valid profiles')

#     return shipDF 


In [188]:
shipDF_filled

,cruiseid,datetime,longitude,latitude,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,...,pco2_ocean,delta_pco2,class1_prob,class2_prob,class3_prob,class4_prob,class5_prob,class6_prob,class_assign,platform
sample_datetag,,,,,,,,,,,,,,,,,,,,,
06AQ20131221_2014010211,06AQ20131221,2014-01-02 11:36:53,-24.64215,-73.725900,1.483947,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,...,343.219907,-42.756075,4.554232e-05,5.047890e-10,7.081806e-21,5.668088e-13,9.999545e-01,5.488803e-54,5.0,socatv2024
06AQ20131221_2014011011,06AQ20131221,2014-01-10 11:59:17,-36.40135,-77.100350,9.499502,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295278,...,367.543164,-17.695680,4.554232e-05,5.047890e-10,7.081806e-21,5.668088e-13,9.999545e-01,5.488803e-54,5.0,socatv2024
06AQ20131221_2014011611,06AQ20131221,2014-01-16 11:48:03,-38.14620,-77.899100,15.491701,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,...,276.631562,-107.323573,1.505514e-07,4.034406e-10,5.907993e-24,3.809673e-13,9.999998e-01,2.364828e-59,5.0,socatv2024
06AQ20131221_2014011707,06AQ20131221,2014-01-17 07:05:05,-38.94590,-77.608000,16.295197,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896516,...,376.886195,-7.583650,1.505514e-07,4.034406e-10,5.907993e-24,3.809673e-13,9.999998e-01,2.364828e-59,5.0,socatv2024
06AQ20131221_2014012211,06AQ20131221,2014-01-22 11:58:20,-28.47890,-74.557400,21.498843,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,...,300.131879,-90.411353,9.494324e-15,3.280363e-10,1.747610e-32,1.725277e-13,1.000000e+00,6.674258e-76,5.0,socatv2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PAT520230724_2023073012,PAT520230724,2023-07-30 12:49:30,163.00795,-39.536095,3497.534375,5905508_id071,2023-08-01,-39.699600,163.253800,2.079502,...,371.927784,-38.513801,2.446198e-250,9.999382e-01,6.167013e-05,1.422797e-07,8.096493e-99,7.654405e-25,2.0,socatv2024
PAT520230724_2023073114,PAT520230724,2023-07-31 14:31:00,154.23600,-37.704690,3498.604861,5906152_id108,2023-07-30,-37.819290,152.754590,1.312465,...,366.040624,-45.320934,0.000000e+00,1.000000e+00,2.393335e-11,6.910590e-09,4.511605e-139,1.922191e-31,2.0,socatv2024
PAT520230724_2023080111,PAT520230724,2023-08-01 11:59:00,149.00550,-38.579580,3499.499306,5905515_id120,2023-07-28,-38.527290,150.643725,3.723206,...,367.773796,-43.689459,1.344386e-311,9.999998e-01,1.898954e-09,2.466102e-07,4.433896e-126,1.019764e-23,2.0,socatv2024


# Optional save


In [ ]:
save_path

# P2_socat-trainvalDF_class-gapfilled_

'../working-vars/regression/P2-clustered/pc8_gmm6/'

In [189]:
shipDF_filled['platform'] = np.tile('socatv2024', len(shipDF_filled))
save = True
if save:
    shipDF_filled.to_csv(save_path + 'P2_socat-trainvalDF_class-gapfilled_' + pcm_params + '_acc' + '20260211' + '.csv')
    print('Saved floatDF_filled to ' + save_path + 'P2_bgcArgo-trainvalDF_class-gapfilled_' + pcm_params + '_acc' + '20260611' + '.csv')
    print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Saved floatDF_filled to ../working-vars/regression/P2-clustered/pc8_gmm6/P2_bgcArgo-trainvalDF_class-gapfilled_pc8_gmm6_acc20260611.csv
2026-02-12 13:31:00


In [175]:
floatDF_filled['platform'] = np.tile('bgcArgo', len(floatDF_filled))
testDF_filled['platform'] = np.tile('bgcArgo', len(testDF_filled))
coreDF_filled['platform'] = np.tile('coreArgo', len(coreDF_filled))
shipDF_filled['platform'] = np.tile('socatv2024', len(shipDF_filled))
                                                   

/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_91237/3524315583.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coreDF_filled['platform'] = np.tile('coreArgo', len(coreDF_filled))


In [177]:
save = True
if save:
    floatDF_filled.to_csv(save_path + 'P2_bgcArgo-trainvalDF_class-gapfilled_' + pcm_params + '_acc' + datetag + '.csv')
    testDF_filled.to_csv(save_path + 'P2_bgcArgo-testDF_class-gapfilled_' + pcm_params + '_acc' + datetag + '.csv')
    coreDF_filled.to_csv(save_path + 'P2_coreArgo-applicationDF_class-gapfilled_' + pcm_params + '_acc' + datetag + '.csv')
    shipDF_filled.to_csv(save_path + 'P2_socat-trainvalDF_class-gapfilled_' + pcm_params + '_acc' + datetag + '.csv')

    print('Saved floatDF_filled to ' + save_path + 'P2_bgcArgo-trainvalDF_class-gapfilled_' + pcm_params + '_acc' + datetag + '.csv')
    print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Saved floatDF_filled to ../working-vars/regression/P2-clustered/pc8_gmm6/P2_bgcArgo-trainvalDF_class-gapfilled_pc8_gmm6_acc20260211.csv
2026-02-11 22:34:52


In [ ]:
# floatDF_filled = pd.read_csv(save_path + 'P2_bgcArgo-trainvalDF_class-gapfilled_' + pcm_params + '_acc' + datetag + '.csv', index_col=0)
# testDF_filled = pd.read_csv(save_path + 'P2_bgcArgo-testDF_class-gapfilled_' + pcm_params + '_acc' + datetag + '.csv', index_col=0)
# coreDF_filled = pd.read_csv(save_path + 'P2_coreArgo-applicationDF_class-gapfilled_' + pcm_params + '_acc' + datetag + '.csv', index_col=0)
# shipDF_filled = pd.read_csv(save_path + 'P2_socat-trainvalDF_class-gapfilled_' + pcm_params + '_acc' + datetag + '.csv', index_col=0)
